# Voice-TTS GPU Worker (Google Colab)

Run the real Qwen3-TTS inference worker on a Colab T4 GPU. Point it at your
local backend via ngrok.

**Before running:**

1. On your local machine, start the backend and expose it with ngrok:
   ```
   cd Voice-Studio/backend
   $env:WORKER_TOKEN="dev-worker-token"; $env:DEV_LOGIN="1"; $env:DEFAULT_JOB_BACKEND="qwen"; $env:FRONTEND_URL="http://localhost:5173"
   python -m uvicorn app.main:app --host 0.0.0.0 --port 8000
   # in another terminal:
   ngrok http 8000
   ```
2. In Colab: **Runtime → Change runtime type → Hardware accelerator: T4 GPU**.
3. Fill in the two values in cell 2 before running.

**Supported job types** (set via `QWEN_MODEL_*` env vars below):

| Env var | Default checkpoint | Job types |
|---------|------------------|----------|
| `QWEN_MODEL_DESIGN` | `Qwen/Voice-TTS-12Hz-1.7B-VoiceDesign` | Voice Design |
| `QWEN_MODEL_BASE` | `Qwen/Qwen3-TTS-12Hz-1.7B-Base` | Clone Prompt, Narration |
| `QWEN_MODEL_CUSTOM_VOICE` | `Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice` | Built-in Voices (Phase 3A) |

In [ ]:
# ============================================================
# 1) Clone the repo and install worker deps
# ============================================================
!git clone https://github.com/016HaiderAli/Voice-Studio.git /content/Voice-Studio
%cd /content/Voice-Studio/worker

!pip install -r requirements.txt -r requirements-qwen.txt

In [ ]:
# ============================================================
# 2) Configure the worker (EDIT THE FIRST TWO VALUES)
# ============================================================
import os

# Paste your current ngrok URL here. It changes unless you have a reserved domain.
os.environ["BACKEND_URL"] = "https://YOUR-NGROK-DOMAIN.ngrok-free.dev"

# MUST match the WORKER_TOKEN in your local backend/.env
os.environ["WORKER_TOKEN"] = "dev-worker-token"

# Run the Qwen backend (not the mock backend).
os.environ["WORKER_BACKEND"] = "qwen"

# Optional: override model checkpoints.
# By default the worker uses the 1.7B variants for Voice Design and Narration,
# and the 0.6B CustomVoice model for Built-in Voices (Phase 3A).
# Uncomment and edit lines below to use different checkpoints:
# os.environ["QWEN_MODEL_DESIGN"]       = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
# os.environ["QWEN_MODEL_BASE"]          = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
# os.environ["QWEN_MODEL_CUSTOM_VOICE"]  = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"
# os.environ["QWEN_DEVICE"]             = "cuda:0"  # change GPU device if needed
# os.environ["QWEN_DTYPE"]              = "bfloat16" # or "float16" for wider compatibility

print("BACKEND_URL:",          os.environ["BACKEND_URL"])
print("WORKER_TOKEN set:",      bool(os.environ["WORKER_TOKEN"]))
print("WORKER_BACKEND:",        os.environ["WORKER_BACKEND"])
print("QWEN_MODEL_CUSTOM_VOICE:", os.environ.get("QWEN_MODEL_CUSTOM_VOICE", "(default)"))

In [ ]:
# ============================================================
# 3) Run the worker (this cell blocks until interrupted)
# ============================================================
!python -m qwen_tts_worker.main --backend qwen

In [ ]:
# # 1. Update/pull the renamed repository into /content/Voice-Studio
# !git clone https://github.com/016HaiderAli/Voice-Studio.git /content/Voice-Studio || (cd /content/Voice-Studio && git pull)

# # 2. Change directory into the worker folder
# %cd /content/Voice-Studio/worker

# # 3. Install missing GPU worker dependencies (includes qwen-tts==0.1.1)
# !pip install -r requirements-qwen.txt

# # 4. Set environment variables and run worker
# import os
# os.environ["BACKEND_URL"] = "https://campsite-citric-snort.ngrok-free.dev"
# os.environ["WORKER_TOKEN"] = "dev-worker-token"
# os.environ["WORKER_BACKEND"] = "qwen"
# os.environ["QWEN_MODEL_CUSTOM_VOICE"] = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"

# !python -m qwen_tts_worker.main --backend qwen